<a href="https://colab.research.google.com/github/cafauzi13/Sentiment-Analysis-M-Paspor-PBA/blob/main/Week3_BoW_Regex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook Pelatihan Model Analisis Sentimen - Ulasan M-Paspor



## I. SETUP LINGKUNGAN DAN IMPOR LIBRARY


### 1.1. Instalasi Library yang Diperlukan

In [1]:
# 1. INSTALASI LIBRARY ]
!pip install pandas numpy scikit-learn tensorflow keras langdetect Sastrawi google-play-scraper nltk textblob

import pandas as pd
import numpy as np
import re
import string
import os
import time
import datetime
import requests
import joblib
import pickle
from pathlib import Path

# Library Scrapping
from google_play_scraper import Sort, reviews, app

# Library NLP & Preprocessing
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory
from textblob import TextBlob

# Download data pendukung NLTK
nltk.download('punkt')
nltk.download('punkt_tab')

# Library Visualisasi (Dari Template Ibu)
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
%matplotlib inline
%config InlineBackend.figure_format='retina'
plt.style.use('seaborn-v0_8')
plt.rcParams["figure.figsize"] = (15,10)

# Library ML & Evaluasi (Dari Template Ibu)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# Setup Reproduksibilitas
DetectorFactory.seed = 42

print("✅ Setup Library Gabungan Selesai. Siap Beraksi!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.9 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=a7182c0a7426e99f7277355ddbdecbaa29e27de1aee353e815a36dee42f1c2b3
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Setup Library Gabungan Selesai. Siap Beraksi!


### 1.2. Load Referensi

In [2]:
# --- 1. KAMUS ALAY (Normalisasi) ---
url_alay = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
df_alay = pd.read_csv(url_alay)
alay_dict = dict(zip(df_alay['slang'], df_alay['formal']))

# --- 2. LEXICON INSET (Labeling) ---
# Kita ambil yang positif dan negatif
url_inset_pos = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
url_inset_neg = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

df_inset_pos = pd.read_csv(url_inset_pos, sep='\t')
df_inset_neg = pd.read_csv(url_inset_neg, sep='\t')
# Gabungkan jadi satu dictionary untuk cek skor kata
inset_dict = dict(zip(df_inset_pos['word'], df_inset_pos['weight']))
inset_dict.update(dict(zip(df_inset_neg['word'], df_inset_neg['weight'])))

# --- 3. COMBINED STOPWORDS (Pembersihan) ---
url_stopwords = "https://raw.githubusercontent.com/louisowen6/NLP_bahasa_resources/master/combined_stop_words.txt"
response = requests.get(url_stopwords)
# Simpan dalam bentuk set agar pencarian kata lebih cepat
combined_stopwords = set(response.text.splitlines())

print(f"✅ Kamus Alay: {len(alay_dict)} kata")
print(f"✅ InSet Lexicon: {len(inset_dict)} kata")
print(f"✅ Combined Stopwords: {len(combined_stopwords)} kata")

✅ Kamus Alay: 4331 kata
✅ InSet Lexicon: 9074 kata
✅ Combined Stopwords: 675 kata



## II. DATA ACQUISITION (SCRAPPING)



### 2.1. Scrapping ulasan M-Paspor dari Google Play

In [3]:
 from google_play_scraper import Sort, reviews
import pandas as pd

# Scrapping
print("⏳ Menarik ulasan (ID & EN)...")
res_id, _ = reviews('id.go.imigrasi.paspor_online', lang='id', country='id', sort=Sort.NEWEST, count=20000)
res_en, _ = reviews('id.go.imigrasi.paspor_online', lang='en', country='id', sort=Sort.NEWEST, count=20000)

all_data = res_id + res_en
if not all_data:
    print("Gagal menarik data")
else:
    # Ubah ke DataFrame
    df_mpaspor = pd.DataFrame(all_data)

    # Hapus duplikat
    df_mpaspor = df_mpaspor.drop_duplicates(subset=['content', 'at'])

    # 3. Seleksi Kolom (Nama kolom harus pas: content, score, dll)
    cols = ["content", "score", "thumbsUpCount", "reviewCreatedVersion", "at", "replyContent", "repliedAt"]
    df_mpaspor = df_mpaspor.loc[:, cols]

    # 4. Simpan (Menimpa file lama)
    df_mpaspor.to_csv('df_mpaspor_review.csv', index=False)

    print(f"✅ Berhasil! Total unik: {len(df_mpaspor)} ulasan.")
    print(f"Daftar Versi: {df_mpaspor['reviewCreatedVersion'].unique()}")


⏳ Menarik ulasan (ID & EN)...
✅ Berhasil! Total unik: 29205 ulasan.
Daftar Versi: ['7.3.10' None '6.1.0' '7.2.5' '7.2.3' '6.4.0' '6.9.0' '6.9.2' '5.4.7'
 '7.3.7' '7.3.6' '6.9.1' '6.7.0' '7.2.0' '7.1.3' '6.2.4' '5.2.1' '6.6.3'
 '6.3.0' '6.6.2' '6.2.5' '6.2.3' '3.0.2' '6.0.2' '6.0.1' '5.0.3' '4.1.0'
 '4.2.0' '2.9.3']


In [4]:
df_mpaspor.head()

,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt
0,aplikasinya mau daftar tapi slalu ditutup gak ...,1,0,7.3.10,2026-03-30 08:25:23,None,NaT
1,TOLONG DONG..... MASA BERKALI KALI BUKA APLIKA...,1,0,None,2026-03-30 02:11:13,None,NaT
2,daftar aja susah mohon di perbaiki,3,0,7.3.10,2026-03-30 01:43:16,None,NaT
3,percuma daftar ngk bisa juga pas masuk verivik...,1,0,7.3.10,2026-03-29 03:13:11,None,NaT
4,saya coba dulu kalau aplikasi nya bagus dan la...,3,0,None,2026-03-29 03:07:46,None,NaT


### 2.2. Inisiasi Stemmer & Fungsi Preprocessing

In [5]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("Menyiapkan Stemmer Sastrawi...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()
print("✅ Stemmer siap!")

Menyiapkan Stemmer Sastrawi...
✅ Stemmer siap!


In [6]:
def preprocess_v1(text):
    # BASIC CLEANING
    text = str(text).lower() # Case folding
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Hapus URL
    text = re.sub(r'[-+]?[0-9]+', '', text)           # Hapus angka
    text = re.sub(r'[^\w\s]', '', text)               # Hapus tanda baca

    # NORMALISASI ALAY
    words = text.split()
    normalized_words = [alay_dict.get(word, word) for word in words]
    text = " ".join(normalized_words)

    # STEMMING (Mencari akar kata - Sastrawi)
    text = stemmer.stem(text)

    # TOKENISASI
    tokens = word_tokenize(text)

    # STOPWORDS REMOVAL
    tokens = [word for word in tokens if word not in combined_stopwords and len(word) > 1]

    return tokens

In [7]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Coba dulu 100
print("⏳ Memulai Preprocessing & Tokenisasi (Sampel 100 data)...")

# kolom baru 'tokens'
df_mpaspor['tokens'] = df_mpaspor['content'].head(100).apply(preprocess_v1)

print("✅ Selesai!")
print(df_mpaspor[['content', 'tokens']].head(10))

⏳ Memulai Preprocessing & Tokenisasi (Sampel 100 data)...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


✅ Selesai!
                                             content  \
0  aplikasinya mau daftar tapi slalu ditutup gak ...   
1  TOLONG DONG..... MASA BERKALI KALI BUKA APLIKA...   
2                 daftar aja susah mohon di perbaiki   
3  percuma daftar ngk bisa juga pas masuk verivik...   
4  saya coba dulu kalau aplikasi nya bagus dan la...   
5  Terima kasih M Paspor.. Aplikasi ini memudahka...   
6                   tidak bisa pilih tanggal booking   
7     ini tgl untuk ngambil kok GK bisa dipilih pak?   
8  tolong dong kak, saya request timeout mulu ket...   
9                                              bagus   

                                              tokens  
0                   [aplikasi, daftar, tutup, kouta]  
1  [buka, aplikasi, sedia, tanggal, nya, suruh, w...  
2                             [daftar, susah, mohon]  
3  [daftar, pas, masuk, verivikasinya, fungsi, ma...  
4  [coba, aplikasi, nya, bagus, lancar, bintang, ...  
5  [terima, kasih, paspor, aplikasi, mudah

In [9]:
version_counts = df_mpaspor['reviewCreatedVersion'].value_counts().reset_index()
version_counts.columns = ['Version', 'Count']
print(version_counts)

top_bad_versions = df_mpaspor[df_mpaspor['score'] == 1]['reviewCreatedVersion'].value_counts().head(5)
print("\nVersi aplikasi yang paling banyak dikritik (Rating 1):")
print(top_bad_versions)

   Version  Count
0    5.2.1   8692
1    5.4.7   5458
2    6.1.0   2756
3    3.0.2   2038
4    6.7.0   1732
5    5.0.3    669
6    6.9.2    523
7    6.6.3    411
8    6.9.1    396
9    6.4.0    382
10   4.1.0    366
11  7.3.10    330
12   6.2.4    287
13   6.3.0    170
14   2.9.3    146
15   4.2.0    142
16   7.2.0    141
17   6.0.1    125
18   7.2.5    115
19   6.6.2     79
20   7.2.3     68
21   6.9.0     54
22   7.1.3     43
23   6.0.2     41
24   6.2.3     32
25   7.3.7     29
26   7.3.6     24
27   6.2.5     24

Versi aplikasi yang paling banyak dikritik (Rating 1):
reviewCreatedVersion
5.2.1    7352
5.4.7    4730
6.1.0    2064
3.0.2    1738
6.7.0    1400
Name: count, dtype: int64


In [10]:
!pip freeze > requirements.txt

## III. PREPROCESSING TEKS

### 3.1. Tokenization


In [13]:
print("⏳ Memproses 20k data (Stemming & Cleaning)...")

# Jalankan ke semua data
df_mpaspor['tokens'] = df_mpaspor['content'].apply(preprocess_v1)

# Gabung token menjadi string kalimat bersih (untuk BoW & TF-IDF)
df_mpaspor['text_clean'] = df_mpaspor['tokens'].apply(lambda x: ' '.join(x))

# Simpan hasil
df_mpaspor.to_csv('df_mpaspor_tokenized.csv', index=False)
print("✅ Preprocessing selesai dan disimpan ke 'df_mpaspor_tokenized.csv'!")

⏳ Memproses 20k data (Stemming & Cleaning)...
✅ Preprocessing selesai dan disimpan ke 'df_mpaspor_tokenized.csv'!


In [14]:
# Load data
df_siap = pd.read_csv('df_mpaspor_tokenized.csv')

# Pastikan tidak ada kolom kosong akibat proses tadi
df_siap['text_clean'] = df_siap['text_clean'].fillna('')
print(f"✅ Data siap olah: {len(df_siap)} baris")

✅ Data siap olah: 29205 baris


## IV. PREPROCESSING TEKS BAHASA INDONESIA
### 4.1. Fungsi Pembersihan Teks Utama

## V. PERSIAPAN DATA UNTUK PELATIHAN MODEL
*Bagian ini dijalankan setelah Bagian IV selesai dan final_df sudah terbentuk*
### 5.1. Pemisahan Fitur dan Label

### 5.2. Encoding Label

### 5.3. Pembagian Data Pelatihan dan Pengujian
 *Kita perlu dua skema pembagian untuk 3 percobaan: 80/20 dan 70/30*

## VI. EKSTRAKSI FITUR
### 6.1. Tokenizer dan Padding untuk Deep Learning
*Mengubah teks menjadi urutan angka (indeks kata)*

### 6.2. TF-IDF untuk Machine Learning Klasik

## VII. IMPLEMENTASI TRAINING (3 skema)


## VIII. INFERENSI/PREDIKSI